# Tutorial 1: Neural Cellular Automata Basics

**Learning Objectives:**
- Understand what Neural Cellular Automata (NCA) are
- Create and configure an NCA model
- Train an NCA to grow a target pattern from a seed
- Visualize and test trained models
- Understand key hyperparameters

**Time:** 1-2 hours

**Prerequisites:** Basic Python, NumPy, and understanding of neural networks

---

## What are Neural Cellular Automata?

Neural Cellular Automata (NCA) are learnable update rules that:
1. Look at local neighborhoods of cells on a grid
2. Apply a neural network to compute updates
3. Repeat this process to create emergent patterns

Think of it like Conway's Game of Life, but:
- The rules are learned, not hand-coded
- Cells can have many channels (not just alive/dead)
- Updates are continuous (not discrete)

**Key Idea:** By training with gradient descent, we can discover update rules that generate complex patterns!

## Setup

First, let's import the required libraries:

In [ ]:
import sys
sys.path.append('..')  # Add parent directory to path

# Core libraries
import jax
import jax.numpy as jnp
import jax.random as jr
import optax
from einops import rearrange
import matplotlib.pyplot as plt

# NCA-specific imports
from NCA.model.NCA_model import NCA
from NCA.trainer.NCA_trainer import NCA_Trainer
from NCA.trainer.data_augmenter_nca import DataAugmenter
from Common.dataloader.emoji import load_emoji_sequence

print(f"JAX version: {jax.__version__}")
print(f"Devices available: {jax.devices()}")

## Part 1: Creating Your First NCA

Let's create a simple NCA model and understand its components.

In [ ]:
# Configuration
N_CHANNELS = 16  # Number of channels per cell

# Create the model
model = NCA(
    N_CHANNELS=N_CHANNELS,
    KERNEL_STR=["ID", "LAP"],  # Perception kernels: Identity and Laplacian
    ACTIVATION=jax.nn.relu,    # Nonlinearity
    PADDING="CIRCULAR",         # Boundary conditions
    FIRE_RATE=0.5              # Probability of updating each cell
)

print(f"Created NCA model with {N_CHANNELS} channels")
print(f"Model type: {type(model)}")

### Understanding the Parameters

- **N_CHANNELS**: Number of features per cell. First 3-4 are usually RGBA colors, rest are "hidden" features
- **KERNEL_STR**: Which spatial operations to use:
  - `"ID"`: Identity (cell's own value)
  - `"LAP"`: Laplacian (difference from neighbors)
  - `"GRAD"`: Gradient (directional differences)
  - `"AVG"`: Average of neighbors
- **ACTIVATION**: Neural network activation function (ReLU, tanh, etc.)
- **PADDING**: How to handle grid boundaries:
  - `"CIRCULAR"`: Wrap around (torus topology)
  - `"ZEROS"`: Pad with zeros
  - `"REFLECT"`: Mirror boundaries
- **FIRE_RATE**: Stochasticity - not all cells update every step (helps stability)

## Part 2: Running an Untrained NCA

Let's see what happens when we run a random (untrained) NCA:

In [ ]:
# Create initial state: mostly zeros with one active pixel in center
GRID_SIZE = 64
x0 = jnp.zeros((N_CHANNELS, GRID_SIZE, GRID_SIZE))

# Set the center pixel to 1.0 in the first 3 channels (RGB)
center = GRID_SIZE // 2
x0 = x0.at[:3, center-1:center+2, center-1:center+2].set(1.0)

print(f"Initial state shape: {x0.shape}")
print(f"Non-zero values: {jnp.sum(x0 != 0)}")

# Visualize initial state
plt.figure(figsize=(4, 4))
plt.imshow(rearrange(x0[:3], "C H W -> H W C"))
plt.title("Initial Seed (3x3 white square)")
plt.axis('off')
plt.show()

In [ ]:
# Run the NCA for 64 steps
N_STEPS = 64
trajectory = model.run(N_STEPS, x0)

print(f"Trajectory shape: {trajectory.shape}")  # (Time, Channels, Height, Width)

# Visualize every 8th frame
trajectory_rgb = rearrange(trajectory[::8, :3], "T C H W -> H (T W) C")

plt.figure(figsize=(20, 4))
plt.imshow(trajectory_rgb)
plt.title("Untrained NCA Evolution (every 8 steps)")
plt.axis('off')
plt.tight_layout()
plt.show()

As you can see, the untrained NCA produces random, chaotic patterns. Now let's train it!

## Part 3: Training an NCA

We'll train the NCA to grow a specific pattern. Let's use an emoji!

### Step 3.1: Load Training Data

In [ ]:
# Configuration
DOWNSAMPLE = 4  # Reduce image size for faster training
NCA_STEPS = 64  # How many NCA steps to reach target
TRAINING_STEPS = 2000
BATCHES = 2

# Load emoji data (target pattern)
# This loads the image twice to create a sequence [initial, target]
data = load_emoji_sequence(
    ["crab.png", "crab.png"],
    impath_emojis="../demo/demo_data/",
    downsample=DOWNSAMPLE
)

print(f"Loaded data shape: {data.shape}")  # (Batch, Time, Channels, Height, Width)

# Create initial condition: small seed in the center
initial_condition = jnp.array(data[:, :1])  # Copy first frame
W, H = initial_condition.shape[-2:]

# Zero out everything except a small center region
initial_condition = initial_condition.at[0, 0, :, :W//2-2].set(0)
initial_condition = initial_condition.at[0, 0, :, W//2+1:].set(0)
initial_condition = initial_condition.at[0, 0, :, :, :H//2-2].set(0)
initial_condition = initial_condition.at[0, 0, :, :, H//2+1:].set(0)

# Concatenate: [seed, target]
data = jnp.concatenate([initial_condition, data], axis=1)

print(f"Final data shape: {data.shape}")

# Visualize
plt.figure(figsize=(12, 4))
plt.imshow(rearrange(data[0, :, :3], "T C H W -> H (T W) C"))
plt.title("Training Data: [Seed] -> [Target]")
plt.axis('off')
plt.show()

### Step 3.2: Set Up Data Augmentation

Data augmentation helps training by:
- Adding padding around the image
- Adding hidden channels beyond RGB
- Enabling batch processing

In [ ]:
class CustomDataAugmenter(DataAugmenter):
    """Custom data augmenter with padding."""
    
    def data_init(self, SHARDING=None):
        """Initialize and preprocess data."""
        data = self.return_saved_data()
        
        # Duplicate to create multiple training batches
        data = self.duplicate_batches(data, BATCHES)
        
        # Add padding around the edges
        data = self.pad(data, 10)
        
        self.save_data(data)
        return None

### Step 3.3: Create the Trainer

In [ ]:
# Create fresh model for training
model = NCA(
    N_CHANNELS=20,  # A few more channels for training
    KERNEL_STR=["ID", "GRAD", "LAP"],
    ACTIVATION=jax.nn.relu,
    PADDING="CIRCULAR",
    FIRE_RATE=0.5
)

# Create trainer
trainer = NCA_Trainer(
    NCA_model=model,
    data=data,
    DATA_AUGMENTER=CustomDataAugmenter,
    model_filename="tutorial_01_crab",
    MODEL_DIRECTORY="../demo/models/",
    LOG_DIRECTORY="../demo/logs/"
)

print("Trainer created successfully!")

### Step 3.4: Train the Model

This will take a few minutes. The training loop:
1. Runs the NCA from seed for `NCA_STEPS` steps
2. Compares output to target
3. Computes gradients
4. Updates NCA parameters
5. Repeats for `TRAINING_STEPS` iterations

In [ ]:
# Set up optimizer with learning rate schedule
schedule = optax.exponential_decay(
    init_value=1e-4,
    transition_steps=TRAINING_STEPS,
    decay_rate=0.99
)

optimiser = optax.chain(
    optax.scale_by_param_block_norm(),
    optax.nadam(schedule)
)

# Train!
trainer.train(
    t=NCA_STEPS,  # Number of NCA steps per training iteration
    iters=TRAINING_STEPS,
    CLEAR_CACHE_EVERY=TRAINING_STEPS,
    LOOP_AUTODIFF="lax",  # Use "checkpointed" if you get out-of-memory errors
    REGULARISER_COEFFS={
        "intermediate_state": 1.0,  # Match intermediate frames too
        "contigous_growth": 1.0,    # Penalize discontinuous growth
    },
    optimiser=optimiser,
    WARMUP=10,
    wandb_args={
        "project": "NCA_Tutorial",
        "name": "tutorial_01_crab",
        "group": "tutorials"
    },
    LOG_EVERY=100
)

print("Training complete!")

## Part 4: Testing the Trained Model

Now let's see if the trained NCA can grow the pattern!

In [ ]:
# Get the trained model
trained_model = trainer.NCA_model

# Get the initial seed state from the augmented data
x0 = trainer.DATA_AUGMENTER.return_saved_data()[0][0]

# Run the trained NCA
trajectory = trained_model.run(NCA_STEPS, x0)

# Visualize results (every 8 steps)
trajectory_rgb = rearrange(trajectory[::8, :3], "T C H W -> H (T W) C")

plt.figure(figsize=(20, 5))
plt.imshow(trajectory_rgb)
plt.title("Trained NCA Growing Pattern (every 8 steps)")
plt.axis('off')
plt.tight_layout()
plt.show()

## Part 5: Testing Robustness

A key feature of NCA is robustness. Let's test if it can:
1. Regenerate from damage
2. Grow from different initial conditions

### Test 1: Regeneration from Damage

In [ ]:
# Start from the target state, but damage it
x0_damaged = trainer.DATA_AUGMENTER.return_saved_data()[0][1].copy()

# Remove the left half
x0_damaged = x0_damaged.at[:, :, :x0_damaged.shape[2]//2].set(0)

plt.figure(figsize=(4, 4))
plt.imshow(rearrange(x0_damaged[:3], "C H W -> H W C"))
plt.title("Damaged Initial State")
plt.axis('off')
plt.show()

# Can it regenerate?
trajectory_regen = trained_model.run(NCA_STEPS, x0_damaged)

trajectory_rgb = rearrange(trajectory_regen[::8, :3], "T C H W -> H (T W) C")
plt.figure(figsize=(20, 5))
plt.imshow(trajectory_rgb)
plt.title("Regeneration from Damage (every 8 steps)")
plt.axis('off')
plt.tight_layout()
plt.show()

### Test 2: Growing from Different Seed

In [ ]:
# Create a different seed: single pixel instead of 3x3
x0_single = jnp.zeros_like(x0)
center_x, center_y = x0.shape[1]//2, x0.shape[2]//2
x0_single = x0_single.at[:3, center_x, center_y].set(1.0)

plt.figure(figsize=(4, 4))
plt.imshow(rearrange(x0_single[:3], "C H W -> H W C"))
plt.title("Single Pixel Seed")
plt.axis('off')
plt.show()

# Grow from single pixel
trajectory_single = trained_model.run(NCA_STEPS, x0_single)

trajectory_rgb = rearrange(trajectory_single[::8, :3], "T C H W -> H (T W) C")
plt.figure(figsize=(20, 5))
plt.imshow(trajectory_rgb)
plt.title("Growing from Single Pixel (every 8 steps)")
plt.axis('off')
plt.tight_layout()
plt.show()

## Part 6: Saving and Loading Models

In [ ]:
# Save the model
save_path = "../demo/models/tutorial_01_crab.eqx"
trained_model.save(save_path)
print(f"Model saved to {save_path}")

# Load the model
# First create a new model with the same architecture
loaded_model = NCA(
    N_CHANNELS=20,
    KERNEL_STR=["ID", "GRAD", "LAP"],
    ACTIVATION=jax.nn.relu,
    PADDING="CIRCULAR",
    FIRE_RATE=0.5
)

# Then load the weights
loaded_model = loaded_model.load(save_path)
print("Model loaded successfully!")

# Verify it works
trajectory_loaded = loaded_model.run(NCA_STEPS, x0)
print(f"Loaded model output shape: {trajectory_loaded.shape}")

## Part 7: Exercises

Try these exercises to deepen your understanding:

### Exercise 1: Different Kernels
Train an NCA with only `["ID", "AVG"]` kernels. How does it differ from using `["ID", "GRAD", "LAP"]`?

### Exercise 2: Channel Count
Train models with 8, 16, 32, and 64 channels. How does channel count affect:
- Training speed?
- Pattern quality?
- Robustness?

### Exercise 3: Fire Rate
Try fire rates of 0.0, 0.25, 0.5, 0.75, and 1.0. What do you observe?

### Exercise 4: Different Patterns
Train on different images from `demo/demo_data/`. Try both simple and complex patterns.

### Exercise 5: Longer Sequences
Instead of `[seed, target]`, try `[seed, intermediate, target]` to guide growth through specific intermediate states.

## Key Takeaways

1. **NCA are learnable local update rules** that can generate complex global patterns
2. **Key hyperparameters:**
   - Channels: More = more expressive but slower
   - Kernels: Different spatial operations give different inductive biases
   - Fire rate: Adds stochasticity and stability
3. **Training process:**
   - Start from seed state
   - Run NCA for N steps
   - Compare to target
   - Update parameters with gradient descent
4. **Emergent properties:**
   - Robustness to damage
   - Generalization to different seeds
   - Self-organization

## Next Steps

- **Tutorial 2:** Learn about PDE modeling
- **Tutorial 3:** Advanced NCA architectures (multi-species, gated, bounded)
- **Demo notebooks:** Explore `demo/image_morph_*.ipynb` for more examples

## Additional Resources

- [Growing Neural Cellular Automata (Distill)](https://distill.pub/2020/growing-ca/)
- [Equinox Documentation](https://docs.kidger.site/equinox/)
- [JAX Documentation](https://jax.readthedocs.io/)

---

**Congratulations!** You've completed Tutorial 1. You now know how to create, train, and test Neural Cellular Automata!